# Strategy correlation and performance metrics

Given trade-level P&L for several strategies, this notebook

1. rolls trades up to daily P&L per strategy,
2. computes pairwise correlation on the days both strategies traded,
3. converts P&L into returns on a fixed capital base and computes Sharpe, max drawdown, VaR and more with QuantStats,
4. writes an HTML tear sheet per strategy.

The trade log is synthetic.

In [ ]:
import os
from pathlib import Path

# All inputs and outputs live under DATA_DIR: data/ at the repo root by default, whether
# Jupyter was started from the repo root or from notebooks/. Set DATA_DIR to override.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = Path(os.environ.get("DATA_DIR", ROOT / "data"))
OUT_DIR = DATA_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

In [ ]:
import itertools
import numpy as np
import pandas as pd
import quantstats as qs

CAPITAL = 100_000  # P&L is divided by this to get daily returns
REPORT_DIR = OUT_DIR / "tearsheets"
REPORT_DIR.mkdir(exist_ok=True)


def synthetic_trades(n_strategies=6, n_days=750, seed=SEED):
    rng = np.random.default_rng(seed)
    days = pd.bdate_range("2021-01-04", periods=n_days)
    common = rng.normal(0, 1, n_days)
    rows = []
    for k in range(n_strategies):
        load = rng.uniform(0, 0.7)
        trade_days = rng.random(n_days) < rng.uniform(0.2, 0.6)
        for d, c in zip(days[trade_days], common[trade_days]):
            for _ in range(rng.integers(1, 4)):
                pnl = 300 * (load * c + np.sqrt(1 - load**2) * rng.normal()) + rng.normal(20, 5)
                rows.append(dict(strategy=f"strategy_{k + 1}", opened=d + pd.Timedelta(hours=int(rng.integers(1, 22))), profit=pnl))
    return pd.DataFrame(rows)


trades = synthetic_trades()
trades.head()

In [ ]:
daily = (trades.assign(date=trades.opened.dt.normalize())
         .groupby(["date", "strategy"]).profit.sum().unstack())  # NaN = no trade that day
names = list(daily.columns)

rows = []
for a, b in itertools.combinations(names, 2):
    both = daily[[a, b]].dropna()
    rows.append(dict(left=a, right=b, common_days=len(both), corr=both[a].corr(both[b])))
pairs = pd.DataFrame(rows).sort_values("corr", ascending=False)
corr_matrix = pairs.pivot(index="left", columns="right", values="corr")
pairs.head(10)

In [ ]:
returns = daily.fillna(0) / CAPITAL
metrics = pd.DataFrame({
    s: {
        "sharpe": qs.stats.sharpe(returns[s]),
        "sortino": qs.stats.sortino(returns[s]),
        "max_drawdown": qs.stats.max_drawdown(returns[s]),
        "value_at_risk": qs.stats.value_at_risk(returns[s]),
        "avg_return": qs.stats.avg_return(returns[s]),
        "win_rate": qs.stats.win_rate(returns[s]),
    } for s in names
}).T
metrics

In [ ]:
for s in names:
    qs.reports.html(returns[s], title=s, output=str(REPORT_DIR / f"{s}.html"))

with pd.ExcelWriter(OUT_DIR / "strategy_metrics.xlsx") as xw:
    pairs.to_excel(xw, sheet_name="pairwise_corr", index=False)
    corr_matrix.to_excel(xw, sheet_name="corr_matrix")
    metrics.to_excel(xw, sheet_name="metrics")